# RealSense D435 → Cornell-style dataset pipeline

Ноутбук-оркестратор для захвата, разметки, проверки и object-wise split собственного датасета под grasp-сегментацию.


## Что учитывает ноутбук

- использует уже принятый в проекте Cornell-совместимый формат;
- depth сохраняется aligned-to-color и в `float32` TIFF (метры);
- объектные метаданные пишутся в `meta.csv` для object-wise split;
- готовые артефакты совместимы с существующим `grasp_seg.data.cornell`.


In [1]:
from pathlib import Path
import os

MAIN_DIR = 'D:/Libraries/Documents/VScode/HRNet_Grasp_Semantic_Segmentation/'
DATASET_ROOT = Path(f'{MAIN_DIR}captures/my_dataset')
SUBDIR = '01'
OBJECT_NAME = 'example_object'
OBJECT_INSTANCE = '01'

# DEPTH_PRESET = 'highdensity'
# MODE = 'manual'


print('DATASET_ROOT =', DATASET_ROOT.resolve())
print('SUBDIR =', SUBDIR)
print('OBJECT_NAME =', OBJECT_NAME)


DATASET_ROOT = D:\Libraries\Documents\VScode\HRNet_Grasp_Semantic_Segmentation\captures\my_dataset
SUBDIR = 01
OBJECT_NAME = example_object


## 1. Smoke-test камеры
Сначала проверьте, что RealSense-стек работает.


In [4]:
!python D:/Libraries/Documents/VScode/HRNet_Grasp_Semantic_Segmentation/tools/realsense_preview.py --out captures/preview

^C


## 2. Захват с метаданными
Скрипт сохраняет `intrinsics.json`, `capture.log`, `meta.csv` и сцены `pcdNNNN*`.


In [2]:
cmd = (
    f"python {MAIN_DIR}tools/realsense_dataset/capture_session.py --out {DATASET_ROOT} --subdir {SUBDIR} "
    f"--object-name {OBJECT_NAME} --object-instance {OBJECT_INSTANCE}"
)
print(cmd)
!{cmd}

NameError: name 'MAIN_DIR' is not defined

## 3. Разметка grasp-прямоугольников
Открывает OpenCV-интерфейс для создания `cpos/cneg`.


In [5]:
cmd = f"python {MAIN_DIR}tools/realsense_dataset/annotate_cornell.py --root {DATASET_ROOT} --subdir {SUBDIR}"
print(cmd)
!{cmd}

python D:/Libraries/Documents/VScode/HRNet_Grasp_Semantic_Segmentation/tools/realsense_dataset/annotate_cornell.py --root D:\Libraries\Documents\VScode\HRNet_Grasp_Semantic_Segmentation\captures\my_dataset --subdir 01
[annotate] 83 images found in D:\Libraries\Documents\VScode\HRNet_Grasp_Semantic_Segmentation\captures\my_dataset\01
  Click P0 (jaw start) → P1 (jaw end) → P2 (height point)
  s=save  p=switch pos/neg  c=clear clicks  z=undo last rect  n=next  q=quit

[1/83] pcd0000  (pos=11, neg=13)
  Saved and quit.


## 4. Проверка датасета
Проверяет пары RGB/depth и количество pos/neg-annotation.


In [11]:
!python {MAIN_DIR}tools/realsense_dataset/verify_cornell_dataset.py --root {DATASET_ROOT}


images=83
positive_rectangles=1222
negative_rectangles=26
missing_depth=0
saved_summary=D:\Libraries\Documents\VScode\HRNet_Grasp_Semantic_Segmentation\captures\my_dataset\dataset_summary.csv


## 5. Object-wise split
`meta.csv` используется как источник истины по объектам.


In [12]:
!python {MAIN_DIR}tools/realsense_dataset/build_objectwise_split.py --root {DATASET_ROOT} --test-frac 0.2 --seed 0

meta.csv not found. capture_session.py should create it.


## 6. Следующий шаг
После этого датасет можно подключать к существующему Cornell-loader и использовать для fine-tune HRNet-W18.
